# 案例 03 · SLB 健康检查失败，全量 502

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

一次依赖变更后，note-api 的 `/healthz` 因数据库超时返回 500，SLB 连续 2 次探测失败摘除唯一后端，App 全量请求返回 502。应用进程本身存活，直连业务接口也正常。

## 练习目标

- 从 SLB 侧取证：健康状态、后端列表
- 区分“后端真故障”与“探活配置/依赖问题”
- 摘除 → 修复 → 等待健康 → 挂回
- 用 `slb_recover.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-03-slb-502")
lab.reset(start_api=True)
lab.start_slb()
lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_slb_backend_unhealthy()
print("已注入：后端健康检查失败（/healthz 依赖 db_timeout）")


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**SLB 健康状态**

In [ ]:
print(lab.aliyun("slb DescribeHealthStatus --LoadBalancerId lb-bp1xhsslb01"))

**直连后端 /healthz**

In [ ]:
print(lab.curl(lab.api + "/healthz"))

**经 SLB 访问**

In [ ]:
print(lab.curl(lab.slb + "/api/v1/feed"))

**SLB 转发统计**

In [ ]:
print(json.dumps(lab.slb_stats(), ensure_ascii=False, indent=2))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
print("后端状态:", lab.state["slb"]["BackendServers"][0]["HealthStatus"])
print("直连业务接口:", lab.curl(lab.api + "/api/v1/feed").splitlines()[-1])
print("经 SLB:", lab.curl(lab.slb + "/api/v1/feed").splitlines()[-1])
print("判断：直连正常、经 SLB 502 → 问题在健康检查/后端摘除，而非业务代码完全不可用")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
lb = lab.state["slb"]["LoadBalancerId"]
# 1) 摘除异常后端，防止 502 继续放大
lab.aliyun(f"slb RemoveBackendServers --LoadBalancerId {lb} --BackendServers i-bp1xhsapi01")
# 2) 修复后端（重启清理异常依赖状态）
lab.restart_api(port=lab.ports["api"])
# 3) 等健康检查通过后挂回（挂回后列表恢复）
lab.wait_until(lambda: "HTTP_CODE=200" in lab.curl(lab.slb + "/api/v1/feed"), timeout=20, interval=1)
lab.aliyun(f"slb AddBackendServers --LoadBalancerId {lb} --BackendServers i-bp1xhsapi01")
lab.state["slb"]["BackendServers"][0]["HealthStatus"] = "normal"
lab._save()
print("经 SLB:", lab.curl(lab.slb + "/api/v1/feed").splitlines()[-1])


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('经 SLB 返回 200', bool('HTTP_CODE=200' in lab.curl(lab.slb + '/api/v1/feed')))
lab.check('后端状态 normal', bool(lab.state['slb']['BackendServers'][0]['HealthStatus'] == 'normal'))

## 步骤 6 · 自动化沉淀

脚本从 `DescribeHealthStatus` 取证，直连判断故障域，先摘除后端止血，重启恢复后等待健康检查通过并挂回。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.start_slb()
lab.inject_slb_backend_unhealthy()
out = lab.auto('slb_recover.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. 健康检查路径应该检查哪些依赖？为什么不宜直接复用重业务接口？
2. 只有 1 个后端时，摘除等于全站不可用，如何设计高可用？
3. 502、504、503 分别可能对应哪一层的问题？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')